# Théorème de Bayes - KMAXPP05

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from matplotlib.patches import Rectangle

In [ ]:
# widget
style = {'description_width': 'initial'}
s_sens = widgets.FloatSlider(min=0, max=1.0, step=0.001, value=0.96, description='Sensibilité :', readout_format=".3f")
s_spec = widgets.FloatSlider(min=0, max=1.0, step=0.001, value=0.99, description='Spécificité :', readout_format=".3f")
s_prior = widgets.FloatSlider(min=0, max=1.0, step=0.001, value=0.03, description='Prévalence :', readout_format=".3f")

out = widgets.Output(layout=widgets.Layout(height='710px', border='2px solid #2ecc71', border_radius='10px', padding='10px'))

def update_plot(sens, spec, prior):
    # Calculs des probabilités jointes
    p_m_pos = prior * sens             # Vrais Positifs
    p_m_neg = prior * (1 - sens)       # Faux Négatifs
    p_s_neg = (1 - prior) * spec       # Vrais Négatifs
    p_s_pos = (1 - prior) * (1 - spec) # Faux Positifs
    
    evidence_pos = p_m_pos + p_s_pos
    posterior = p_m_pos / evidence_pos if evidence_pos > 0 else 0
    bayes_factor = sens / (1 - spec)

    with out:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(7, 7))
        
        scale = 1000.
        w_m = prior * scale
        w_s = (1 - prior) * scale

        FN = p_m_neg * scale
        VP = p_m_pos * scale
        VN = p_s_neg * scale
        FP = p_s_pos * scale
    
        # Rectangles
        # Infectés (gauche)
        # FN en bas, VP en haut
        r_fn = Rectangle((0, 0), w_m, (1-sens)*scale, fc="#f1c40f", alpha=0.2, label=f"FN: {FN:.0f}")
        r_vp = Rectangle((0, (1-sens)*scale), w_m, sens*scale, fc="#ff7675", alpha=0.9, label=f"VP: {VP:.0f}")
        
        # Non-infectés (droite)
        # VN en bas, FP en haut
        r_vn = Rectangle((w_m, 0), w_s, spec*scale, fc="#2ecc71", alpha=0.2, label=f"VN: {VN:.0f}")
        r_fp = Rectangle((w_m, spec*scale), w_s, (1-spec)*scale, fc="#3498db", alpha=0.9, label=f"FP: {FP:.0f}")
        
        for r in [r_fn, r_vp, r_vn, r_fp]: ax.add_patch(r)

        # Customisation
        ax.set_xlim(0, scale)
        ax.set_ylim(0, scale)
        ax.set_aspect('equal') # Carré parfait
        
        # On trace le "rideau" vertical
        ax.axvline(w_m, color='black', linewidth=2)
        
        # Labels
        ax.set_xticks([w_m/2, w_m + w_s/2])
        ax.set_xticklabels([f'Infectés\n({prior*scale:.0f})', f'Non-infectés\n({(1-prior)*scale:.0f})'], fontweight='bold')
        ax.set_yticks([])
        
        plt.title(f"Diagramme de Bayes (Population Totale Fixe = 1000)\n$P(C|T)$ = {posterior:.2%}; Facteur de Bayes BF10(T) = {bayes_factor:.2f}", fontsize=12)
        plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.10), ncol=2)
        plt.tight_layout()
        plt.show()

interactive_plot = widgets.interactive_output(update_plot, {'sens': s_sens, 'spec': s_spec, 'prior': s_prior})
display(widgets.VBox([widgets.HBox([s_prior, s_sens, s_spec]), out]))

In [ ]:
# Etat
state = {'p_a1': 0.5, 'n_pos': 0, 'n_neg': 0}

# Widget
style = {'description_width': 'initial'}
layout_slider = widgets.Layout(width='320px')

s_prior = widgets.FloatSlider(min=0, max=1, step=0.001, value=0.5, readout_format=".3f",
                              description='A priori $P(A=1)$ :', style=style, layout=layout_slider)
s_lik_pos = widgets.FloatSlider(min=0, max=1, step=0.01, value=0.5, readout_format=".3f",
                                description='Sensibilité $P(C=1|A=1)$ :', style=style, layout=layout_slider)
s_lik_neg = widgets.FloatSlider(min=0, max=1, step=0.01, value=0.5, readout_format=".3f",
                                description='Spécificité (-) $P(C=0|A=0)$ :', style=style, layout=layout_slider)

btn_pos = widgets.Button(description="Critique Positive (+)", button_style='success', layout=widgets.Layout(width='200px'))
btn_neg = widgets.Button(description="Critique Négative (-)", button_style='danger', layout=widgets.Layout(width='200px'))
btn_reset = widgets.Button(description="Réinitialiser", button_style='info')

out = widgets.Output(layout=widgets.Layout(height='550px', border='2px solid #2ecc71', border_radius='10px', padding='10px'))

# Calcul

def render_step(info_maths="Initialisation"):
    with out:
        clear_output(wait=True)
        p1 = state['p_a1']
        p0 = 1 - p1
        
        # Graphique de la fonction de masse
        fig, ax = plt.subplots(figsize=(8, 4.5))
        bars = ax.bar(['Avis Négatif ($A=0$)', 'Avis Positif ($A=1$)'], [p0, p1], 
                      color=['#e74c3c', '#2ecc71'], edgecolor='black', alpha=0.8)
        
        ax.set_ylim(0, 1.15)
        ax.set_title(f"Mise à jour de la conviction (Bayes)", fontsize=14)
        
        for bar in bars:
            h = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2, h + 0.02, f"{h:.4f}", ha='center', weight='bold', size=13)
        
        plt.tight_layout()
        plt.show()
        
        # Encart de texte informatif
        print(f"📊 JOURNAL DE BORD :")
        print(f"Critiques : {state['n_pos']} 👍 | {state['n_neg']} 👎")
        print(info_maths)

def update_bayes(obs):
    # Sauvegarde de l'état
    prior_a1 = state['p_a1']
    prior_a0 = 1 - prior_a1
    
    # Lecture des widgets
    l_pos = s_lik_pos.value
    l_neg = s_lik_neg.value
    
    # Choix de la vraisemblance selon l'observation
    if obs == 1: # Critique positive
        lik_a1 = l_pos             # P(C=1|A=1)
        lik_a0 = 1 - l_neg         # P(C=1|A=0)
        state['n_pos'] += 1
        label = "POSITIVE (+)"
    else: # Critique négative
        lik_a1 = 1 - l_pos         # P(C=0|A=1)
        lik_a0 = l_neg             # P(C=0|A=0)
        state['n_neg'] += 1
        label = "NÉGATIVE (-)"
        
    # Théorème de Bayes
    numerateur = lik_a1 * prior_a1
    denominateur = (lik_a1 * prior_a1) + (lik_a0 * prior_a0)
    
    if denominateur > 0:
        state['p_a1'] = numerateur / denominateur
    
    # Texte de debug
    math_txt = (f"Observation : {label}\n"
                f"Vraisemblances utilisées : P(C|A=1)={lik_a1:.3f}, P(C|A=0)={lik_a0:.3f}\n"
                f"Calcul : ({lik_a1:.3f} * {prior_a1:.3f}) / {denominateur:.3f} = {state['p_a1']:.4f}")
    
    render_step(math_txt)

def reset_all(b=None):
    state['p_a1'] = s_prior.value
    state['n_pos'] = 0
    state['n_neg'] = 0
    render_step(f"Reset effectué. A priori de départ : {s_prior.value}")

# --- 4. LIAISONS ---
btn_pos.on_click(lambda b: update_bayes(1))
btn_neg.on_click(lambda b: update_bayes(0))
btn_reset.on_click(reset_all)
s_prior.observe(lambda c: reset_all(), names='value')

# --- 5. DISPLAY ---
ui = widgets.VBox([
    widgets.HTML("<h2>🧠 Inférence Bayésienne</h2>"),
    widgets.HBox([s_prior, s_lik_pos, s_lik_neg], layout=widgets.Layout(justify_content='space-around')),
    widgets.HBox([btn_pos, btn_neg, btn_reset], layout=widgets.Layout(justify_content='center', gap='40px', margin='10px')),
    out
])

display(ui)
reset_all()